# Repository guide: Historical test-draft annotation aid; do not overwrite frozen references

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


In [ ]:
# Cell 1 — Mount Drive and define project paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "transcription_queue_test_REC052_REC059_mms_drafts.csv"
)

print(PROJECT_ROOT)

Mounted at /content/drive
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm


In [ ]:
# Cell 2 — Load the test-segment metadata

import pandas as pd

df = pd.read_csv(METADATA_PATH)

TEST_RECORDINGS = ["REC052", "REC059"]

test_df = df[
    df["recording_id"].isin(TEST_RECORDINGS)
].copy()

print("Test candidate segments:", len(test_df))

print(
    test_df["recording_id"]
    .value_counts()
)

Test candidate segments: 249
recording_id
REC052    196
REC059     53
Name: count, dtype: int64


In [ ]:
# Cell 3 — Load the MMS-Tachebdant transcription assistant

!pip -q install transformers librosa soundfile

import torch
import librosa

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
)

MODEL_ID = "iukocha/mms-tachebdant-from-tarifit"

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = AutoModelForCTC.from_pretrained(
    MODEL_ID
).to(device)

model.eval()

print("Device:", device)

processor_config.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Device: cuda


In [ ]:
# Cell 4 — Inspect test audio paths

print(
    test_df[
        [
            "segment_id",
            "recording_id",
            "audio_path",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

    segment_id recording_id                                        audio_path
REC052_SEG0001       REC052 data/processed/segments/REC052/REC052_SEG0001.wav
REC052_SEG0002       REC052 data/processed/segments/REC052/REC052_SEG0002.wav
REC052_SEG0003       REC052 data/processed/segments/REC052/REC052_SEG0003.wav
REC052_SEG0004       REC052 data/processed/segments/REC052/REC052_SEG0004.wav
REC052_SEG0005       REC052 data/processed/segments/REC052/REC052_SEG0005.wav
REC052_SEG0006       REC052 data/processed/segments/REC052/REC052_SEG0006.wav
REC052_SEG0007       REC052 data/processed/segments/REC052/REC052_SEG0007.wav
REC052_SEG0008       REC052 data/processed/segments/REC052/REC052_SEG0008.wav
REC052_SEG0009       REC052 data/processed/segments/REC052/REC052_SEG0009.wav
REC052_SEG0010       REC052 data/processed/segments/REC052/REC052_SEG0010.wav


In [ ]:
# Cell 5 — Verify that the test audio files exist

test_df["colab_audio_path"] = test_df["audio_path"].apply(
    lambda p: PROJECT_ROOT / p
)

missing = test_df[
    ~test_df["colab_audio_path"].apply(lambda p: p.exists())
]

print("Test segments:", len(test_df))
print("Missing audio files:", len(missing))

if len(missing):
    print(missing[["segment_id", "audio_path"]].head(20))
else:
    print("✓ All test audio files found")

Test segments: 249
Missing audio files: 0
✓ All test audio files found


In [ ]:
# Cell 6 — Generate MMS-Tachebdant draft transcriptions

from tqdm.auto import tqdm
import torch
import librosa

drafts = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df)
):

    audio, sr = librosa.load(
        row["colab_audio_path"],
        sr=16000,
        mono=True
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(
            input_values
        ).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    transcription = processor.batch_decode(
        predicted_ids
    )[0]

    drafts.append(transcription)

test_df["mms_draft_raw"] = drafts

print(
    test_df[
        ["segment_id", "mms_draft_raw"]
    ].head(10)
)

  0%|          | 0/249 [00:00<?, ?it/s]

          segment_id                                      mms_draft_raw
1474  REC052_SEG0001  a nədhwər ɣawinath ɣa tsmaɣ lmuwḍhuɛa n zəddix...
1475  REC052_SEG0002  mašanəš maš twarix nəš smə  ssəqsi ɣa əsmaḥadh...
1476  REC052_SEG0003  madi lmustəqbəl ḥm ad tssəhwam taman rəkranəšš...
1477  REC052_SEG0004  xrun ydəɣɣ tx ddulaba šan təg, min bayniha, ad...
1478  REC052_SEG0005                                           n priḍi.
1479  REC052_SEG0006  ixəssamathən ad is-yərijen nnisbamuɛayyana n ṣ...
1480  REC052_SEG0007  sijam ijjen n əndima jarasən, wa dh lḥəl n -dd...
1481  REC052_SEG0008  adh sənəksən dhi rəšra, thinan ddula.walakin d...
1482  REC052_SEG0009  məra ixəddəm ɣars llimkanyat ad iɣza adh raḥ a...
1483  REC052_SEG0010  wərni ɣa ja limkiitmanəfṭən ad ysəɣ baš ḥətta ...


In [ ]:
# Cell 7 — Normalize MMS drafts and safely export the transcription queue

import re
import unicodedata
from collections import Counter


# Final V1.2 alphabet
FINAL_LETTERS = set(
    "abcdefghijklmn"
    "pqrstuvwxyz"
)

FINAL_LETTERS.discard("o")

FINAL_LETTERS.update({
    "ɛ",
    "ɣ",
    "ʷ",
    "ḍ",
    "ḥ",
    "ṭ",
})

ALLOWED = FINAL_LETTERS | {" "}


def normalize_mms_draft(text):

    text = unicodedata.normalize(
        "NFC",
        str(text).lower()
    )

    # MMS/Tachebdant → project conventions
    replacements = {
        "ə": "e",
        "š": "c",
        "ā": "a",

        # Final V1.2 rules
        "ṣ": "s",
        "ẓ": "z",
        "ṛ": "r",
        "ǧ": "dj",
        "3": "ɛ",
        "o": "u",
        "\u0323": "",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove punctuation
    text = "".join(
        " " if unicodedata.category(ch).startswith("P")
        else ch
        for ch in text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


test_df["mms_draft_normalized"] = (
    test_df["mms_draft_raw"]
    .apply(normalize_mms_draft)
)


# --------------------------------------------------------
# Flag symbols that still do not belong to V1.2
# --------------------------------------------------------

def unexpected_characters(text):
    return sorted(
        {
            ch
            for ch in text
            if ch not in ALLOWED
        }
    )


test_df["mms_normalization_warning"] = (
    test_df["mms_draft_normalized"]
    .apply(unexpected_characters)
)

warning_rows = test_df[
    test_df["mms_normalization_warning"]
    .apply(len)
    .gt(0)
]

print(
    "Rows containing unresolved MMS symbols:",
    len(warning_rows)
)

if len(warning_rows):
    display(
        warning_rows[
            [
                "segment_id",
                "mms_draft_raw",
                "mms_draft_normalized",
                "mms_normalization_warning",
            ]
        ].head(30)
    )


# --------------------------------------------------------
# Columns for your manual test transcription
# --------------------------------------------------------

if "final_transcription" not in test_df.columns:
    test_df["final_transcription"] = ""

if "manual_review" not in test_df.columns:
    test_df["manual_review"] = ""

if "transcription_notes" not in test_df.columns:
    test_df["transcription_notes"] = ""


# --------------------------------------------------------
# Do not accidentally overwrite manual work
# --------------------------------------------------------

if OUTPUT_PATH.exists():

    print(
        "\nWARNING: Output CSV already exists."
    )

    print(
        "Not overwriting it automatically "
        "to protect manual transcriptions."
    )

else:

    test_df.to_csv(
        OUTPUT_PATH,
        index=False,
        encoding="utf-8"
    )

    print("\nSaved:")
    print(OUTPUT_PATH)


display(
    test_df[
        [
            "segment_id",
            "mms_draft_raw",
            "mms_draft_normalized",
            "mms_normalization_warning",
        ]
    ].head(10)
)

Rows containing unresolved MMS symbols: 0

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/transcription_queue_test_REC052_REC059_mms_drafts.csv


,segment_id,mms_draft_raw,mms_draft_normalized,mms_normalization_warning
1474,REC052_SEG0001,a nədhwər ɣawinath ɣa tsmaɣ lmuwḍhuɛa n zəddix...,a nedhwer ɣawinath ɣa tsmaɣ lmuwḍhuɛa n zeddix...,[]
1475,REC052_SEG0002,mašanəš maš twarix nəš smə ssəqsi ɣa əsmaḥadh...,macanec mac twarix nec sme sseqsi ɣa esmaḥadh ...,[]
1476,REC052_SEG0003,madi lmustəqbəl ḥm ad tssəhwam taman rəkranəšš...,madi lmusteqbel ḥm ad tssehwam taman rekranecc...,[]
1477,REC052_SEG0004,"xrun ydəɣɣ tx ddulaba šan təg, min bayniha, ad...",xrun ydeɣɣ tx ddulaba can teg min bayniha ad t...,[]
1478,REC052_SEG0005,n priḍi.,n priḍi,[]
1479,REC052_SEG0006,ixəssamathən ad is-yərijen nnisbamuɛayyana n ṣ...,ixessamathen ad is yerijen nnisbamuɛayyana n s...,[]
1480,REC052_SEG0007,"sijam ijjen n əndima jarasən, wa dh lḥəl n -dd...",sijam ijjen n endima jarasen wa dh lḥel n ddul...,[]
1481,REC052_SEG0008,"adh sənəksən dhi rəšra, thinan ddula.walakin d...",adh seneksen dhi recra thinan ddula walakin dy...,[]
1482,REC052_SEG0009,məra ixəddəm ɣars llimkanyat ad iɣza adh raḥ a...,mera ixeddem ɣars llimkanyat ad iɣza adh raḥ a...,[]
1483,REC052_SEG0010,wərni ɣa ja limkiitmanəfṭən ad ysəɣ baš ḥətta ...,werni ɣa ja limkiitmanefṭen ad yseɣ bac ḥetta ...,[]
